# Laboratorio de Procesamiento de Videos en Español

Este notebook está diseñado para trabajar con videos de AWS *"This is My Architecture"* grabados en **español** u otros idiomas distintos al inglés.

El flujo del laboratorio realiza las siguientes tareas:
1. **Preparación de Datos:** Descarga de video en `data/raw` (si no existe), extracción local de audio en WAV y extracción de keyframes en `whiteboard_selection_lab/frames_new/VIDEO_ID`.
2. **Detección Automática de Idioma y Transcripción:** Carga del modelo Whisper `turbo`, análisis de los primeros 30 segundos del audio para auto-detectar el idioma y transcripción total guardada en local.
3. **Selección Inteligente de Pizarra:** Ejecución del selector inteligente (`frame_selector`) sobre los frames locales para encontrar la mejor toma libre de oclusión humana.
4. **Construcción y Comparación de Grafos (Cache):** Mapeo de los resultados de análisis de visión en cache (Standard y Parsimonious) a grafos de NetworkX y comparación directa contra el Ground Truth.
5. **Evaluación de Prompts e Integración de Idiomas (Gemini API):** Ejecución de un experimento en vivo con la API de Gemini Vision bajo prompts adaptados para traducir notas y descripciones del español al inglés.
6. **Comparación Detallada de Rendimiento:** Resumen interactivo en una tabla Pandas comparando el rendimiento de los tres grafos (Standard, Parsimonious y el de Prueba) contra el Ground Truth.

In [8]:
import cv2
import shutil
import os
import json
import sys
import torch
from pathlib import Path
import whisper

# ==========================================
# 1. CONFIGURACIÓN DEL VIDEO EN ESPAÑOL
# ==========================================
# Ejemplos de videos en español en la base de datos:
# - "CsD5bmM6mpY" (Linke - Syntax Company: Building a Business Continuity Framework for SAP)
# - "l0mvXogANE4" Smadex: Serving Billions of Ads with Amazon ElastiCache for Redis
# - "x76BIV88j_M" Cabify: Building a Massively Scalable Ride-Sharing Mobile App with Containers

VIDEO_ID = "l0mvXogANE4"

PROJECT_ROOT = Path('/home/stemjara/Projects/AWS-Architecture')
LAB_DIR = PROJECT_ROOT / 'whiteboard_selection_lab'

# Carpetas de trabajo locales en whiteboard_selection_lab
FRAMES_NEW_DIR = LAB_DIR / 'frames_new' / VIDEO_ID
TRANSCRIPTIONS_DIR = LAB_DIR / 'transcriptions'
LAB_WORKSPACE = LAB_DIR / 'lab_workspace' / VIDEO_ID
AUDIO_DIR = LAB_WORKSPACE / 'audio'

for d in (FRAMES_NEW_DIR, TRANSCRIPTIONS_DIR, LAB_WORKSPACE, AUDIO_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"🚀 Iniciando preparación para el video en español: {VIDEO_ID}")

🚀 Iniciando preparación para el video en español: l0mvXogANE4


In [9]:
# ==========================================
# 2. DESCARGA Y EXTRACCIÓN DE AUDIO
# ==========================================
raw_video_path = PROJECT_ROOT / 'data' / 'raw' / f"{VIDEO_ID}.mp4"
audio_path = AUDIO_DIR / f"{VIDEO_ID}.wav"

# Descargar el video si no existe localmente
if not raw_video_path.exists():
    print(f"📥 Video {VIDEO_ID}.mp4 no encontrado en raw. Descargándolo...")
    sys.path.append(str(PROJECT_ROOT))
    from scripts.core.downloader import download_video
    video_url = f"https://www.youtube.com/watch?v={VIDEO_ID}"
    download_video(video_url, output_dir=PROJECT_ROOT / 'data' / 'raw')
else:
    print(f"✅ Video encontrado en: {raw_video_path}")

# Extraer audio en PCM WAV 16kHz mono
if not audio_path.exists():
    print(f"🔊 Extrayendo audio (FFmpeg) a {audio_path}...")
    os.system(f"ffmpeg -y -i {raw_video_path} -vn -acodec pcm_s16le -ar 16000 -ac 1 {audio_path}")
    print("✅ Audio extraído.")
else:
    print(f"✅ El audio ya existe en: {audio_path}")

✅ Video encontrado en: /home/stemjara/Projects/AWS-Architecture/data/raw/l0mvXogANE4.mp4
🔊 Extrayendo audio (FFmpeg) a /home/stemjara/Projects/AWS-Architecture/whiteboard_selection_lab/lab_workspace/l0mvXogANE4/audio/l0mvXogANE4.wav...
✅ Audio extraído.


ffmpeg version n8.1.1 Copyright (c) 2000-2026 the FFmpeg developers
  built with gcc 16.1.1 (GCC) 20260430
  configuration: --prefix=/usr --disable-debug --disable-static --disable-stripping --enable-amf --enable-avisynth --enable-cuda-llvm --enable-lto --enable-fontconfig --enable-frei0r --enable-gmp --enable-gnutls --enable-gpl --enable-ladspa --enable-lcms2 --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libdav1d --enable-libdrm --enable-libdvdnav --enable-libdvdread --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgsm --enable-libharfbuzz --enable-libiec61883 --enable-libjack --enable-libjxl --enable-libmodplug --enable-libmp3lame --enable-libopencore_amrnb --enable-libopencore_amrwb --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libplacebo --enable-libpulse --enable-librav1e --enable-librsvg --enable-librubberband --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-l

In [11]:
# ==========================================
# 3. DETECCIÓN AUTOMÁTICA DE IDIOMA Y TRANSCRIPCIÓN (Whisper)
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🧠 Cargando modelo Whisper 'turbo' en {device}...")
model = whisper.load_model("turbo", device=device)

# Detección automática de idioma usando los primeros 30 segundos de audio
print("🕵️ Detectando idioma del audio...")
audio = whisper.load_audio(str(audio_path))
audio = whisper.pad_or_trim(audio)
# Usar model.dims.n_mels asegura que siempre coincida, sin importar qué modelo cargues
mel = whisper.log_mel_spectrogram(audio, n_mels=model.dims.n_mels).to(model.device)

_, probs = model.detect_language(mel)
detected_lang = max(probs, key=probs.get)
print(f"✨ Idioma detectado automáticamente: '{detected_lang}' (Probabilidad: {probs[detected_lang]:.4f})")

# Transcribir con Whisper usando el idioma detectado
print("📝 Generando transcripción completa con Whisper...")
result = model.transcribe(str(audio_path), language=detected_lang)

# Formatear segmentos con marca de tiempo
sys.path.append(str(PROJECT_ROOT))
from scripts.core.transcriber import get_timestamped_segments
segments = get_timestamped_segments(result)

# Guardar en el entorno del lab y en cache global raw
transcript_file = TRANSCRIPTIONS_DIR / f"{VIDEO_ID}_transcript.json"
with open(transcript_file, "w", encoding="utf-8") as f:
    json.dump(segments, f, indent=2, ensure_ascii=False)

global_transcript = PROJECT_ROOT / 'data' / 'raw' / f"{VIDEO_ID}_transcript.json"
shutil.copy2(transcript_file, global_transcript)

print(f"✅ Transcripción guardada en local del lab: {transcript_file}")
print(f"✅ Transcripción copiada a cache global: {global_transcript}")

🧠 Cargando modelo Whisper 'turbo' en cuda...
🕵️ Detectando idioma del audio...
✨ Idioma detectado automáticamente: 'es' (Probabilidad: 0.9988)
📝 Generando transcripción completa con Whisper...
✅ Transcripción guardada en local del lab: /home/stemjara/Projects/AWS-Architecture/whiteboard_selection_lab/transcriptions/l0mvXogANE4_transcript.json
✅ Transcripción copiada a cache global: /home/stemjara/Projects/AWS-Architecture/data/raw/l0mvXogANE4_transcript.json


In [12]:
# ==========================================
# 4. EXTRACCIÓN DINÁMICA DE KEYFRAMES
# ==========================================
cap = cv2.VideoCapture(str(raw_video_path))
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
umbral_80_porciento = int(total_frames * 0.80)

intervalo_lento = int(fps * 10)  # Cada 10 segundos
intervalo_rapido = int(fps * 1)   # Cada 1 segundo

frame_count = 0
saved_count = 0

print(f"🎞️ Extrayendo keyframes... (FPS: {fps:.2f}, Total frames: {total_frames})")

# Limpiar frames anteriores del lab para este video
for f in FRAMES_NEW_DIR.glob('*.jpg'):
    f.unlink()

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    intervalo_actual = intervalo_lento if frame_count < umbral_80_porciento else intervalo_rapido
    if frame_count % intervalo_actual == 0:
        cv2.imwrite(str(FRAMES_NEW_DIR / f"{VIDEO_ID}_frame_{frame_count}.jpg"), frame)
        saved_count += 1
    frame_count += 1
cap.release()

print(f"✅ Se guardaron {saved_count} frames en: {FRAMES_NEW_DIR}")

🎞️ Extrayendo keyframes... (FPS: 24.00, Total frames: 8110)
✅ Se guardaron 95 frames en: /home/stemjara/Projects/AWS-Architecture/whiteboard_selection_lab/frames_new/l0mvXogANE4


In [13]:
# ==========================================
# 5. SELECCIÓN DE PIZARRA INTELIGENTE
# ==========================================
sys.path.append(str(PROJECT_ROOT))
from scripts.core.frame_selector import select_best_frame

print("🔍 Ejecutando el Frame Selector inteligente...")
# Usamos la carpeta local frames_new del lab
res_selector = select_best_frame(VIDEO_ID, frames_dir=LAB_DIR / 'frames_new', debug=True)

# Copiar a lab_workspace para tener una referencia local cómoda de la pizarra ganadora
best_frame_path = Path(res_selector['best_frame'])
local_best_whiteboard = LAB_WORKSPACE / "best_whiteboard.jpg"
shutil.copy2(best_frame_path, local_best_whiteboard)

print(f"🏆 Pizarra seleccionada: {best_frame_path.name}")
print(f"📁 Copia local de pizarra ganadora guardada en: {local_best_whiteboard}")

🔍 Ejecutando el Frame Selector inteligente...


🔍 Frame Selection for l0mvXogANE4 (95 frames total)

🚀 Procesando 76 frames (Último 80% del video)...

✓ Selected: l0mvXogANE4_frame_7824.jpg (score=29.848, edge=0.030, occ=2.3%, bonus=5.00, iconos=5)

✓ Saved → 
/home/stemjara/Projects/AWS-Architecture/whiteboard_selection_lab/frames_new/l0mvXogANE4_pizarra/best_whiteboard.jp
g

✓ Copied to bad_whiteboard → 
/home/stemjara/Projects/AWS-Architecture/whiteboard_selection_lab/bad_whiteboard/l0mvXogANE4.jpg

Debug data saved → 
/home/stemjara/Projects/AWS-Architecture/whiteboard_selection_lab/frames_new/l0mvXogANE4_pizarra/frame_selection_de
bug.json

🏆 Pizarra seleccionada: best_whiteboard.jpg
📁 Copia local de pizarra ganadora guardada en: /home/stemjara/Projects/AWS-Architecture/whiteboard_selection_lab/lab_workspace/l0mvXogANE4/best_whiteboard.jpg


## 7. DEFINICIÓN DE PROMPTS Y EJECUCIÓN CON GEMINI API (Con Pydantic)

Version en Ingles para trabajar de manera general


In [17]:
# ==========================================
# 7. DEFINICIÓN DE PROMPTS Y EJECUCIÓN CON GEMINI API (Con Pydantic)
# ==========================================
import base64
import json
import csv
import re
import networkx as nx
from google import genai
from pydantic import BaseModel
from config.settings import GEMINI_API_KEY, GEMINI_MODEL
from scripts.core.graph_builder import create_graph_from_cloudscape_json

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY no configurado. Revisa tu archivo .env")

client = genai.Client(api_key=GEMINI_API_KEY)

# 1. Cargar y codificar imagen de pizarra local en base64
ws_whiteboard = LAB_WORKSPACE / "best_whiteboard.jpg"
if not ws_whiteboard.exists():
    # Intenta copiar desde frames_new/video_id_pizarra/best_whiteboard.jpg
    pizarra_dir = LAB_DIR / 'frames_new' / f"{VIDEO_ID}_pizarra"
    shutil.copy2(pizarra_dir / "best_whiteboard.jpg", ws_whiteboard)

image_bytes = ws_whiteboard.read_bytes()
image_b64 = base64.b64encode(image_bytes).decode("utf-8")

# 2. Cargar transcripción local
transcript_file = TRANSCRIPTIONS_DIR / f"{VIDEO_ID}_transcript.json"
with open(transcript_file, "r", encoding="utf-8") as f:
    segments = json.load(f)
transcript_text = " ".join(s.get("text", "").strip() for s in segments)

# 3. Cargar catálogo de servicios y actores
services_csv = PROJECT_ROOT / "data" / "cloudscape_gt" / "services.csv"
aws_services_list, user_actors_list = [], []
with open(services_csv, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        name = row.get("name", "").strip()
        capability = row.get("capability", "").strip().lower()
        if not name:
            continue
        if capability == "user" or name.startswith("User"):
            user_actors_list.append(name)
        else:
            aws_services_list.append(name)

aws_services_str = ", ".join(sorted(aws_services_list))
user_actors_str = ", ".join(sorted(user_actors_list))

# ==========================================
# ESQUEMAS PYDANTIC (Garantía de JSON perfecto)
# ==========================================
class Entity(BaseModel):
    service: str
    name: str
    type: str
    rationale: str

class VisualConnection(BaseModel):
    source_label: str
    target_label: str
    arrow_direction: str
    description: str

class WorldModelSchema(BaseModel):
    entities: list[Entity]
    visual_connections: list[VisualConnection]

# ==========================================
# ESQUEMAS PYDANTIC - FASE 2 (Grafo Final)
# ==========================================
class GraphMetadata(BaseModel):
    name: str
    link: str
    categories: str
    graph_usable: bool
    notes: str

class Node(BaseModel):
    id: str
    service: str
    name: str
    notes: str

class Edge(BaseModel):
    source: str
    target: str
    flow_id: int
    seq: str
    type: str
    notes: str

class FinalArchitectureSchema(BaseModel):
    step_by_step_reasoning: str
    graph: GraphMetadata
    nodes: list[Node]
    edges: list[Edge]

# 4. Definición de prompts (Modeler y Planner) con soporte de traducción al inglés
MFR_STAGE_1_PROMPT_TEMPLATE = """You are an expert in visual extraction of cloud architectures. Your task is to analyze the provided whiteboard diagram and audio transcript to build a structured "World Model" representing the components and their visual groupings.

Do NOT generate the final graph. Focus strictly on identifying the present entities, how they are visually grouped, and what lines connect them.

## ENTITY IDENTIFICATION AND GROUPING RULES:
1. Identify all active systems, databases, cloud services, and actors VISIBLE on the whiteboard.
2. Identify BOXES or dashed containers. If a service (e.g., SAP) is drawn inside another service or box (e.g., EC2), register both and strictly note this containment in the "rationale".
3. Use valid AWS services from this list: <AWS_SERVICES_PLACEHOLDER>.
4. Identify actors/users from this list: <USER_ACTORS_PLACEHOLDER>. Map on-premises/external systems to "ThirdParty" and actors to "User".
5. Do NOT include transient files, packages, or disk images (like AMIs) as entities.

## VISUAL CONNECTION RULES:
1. Register ALL lines or arrows explicitly drawn between entities on the whiteboard.
2. Note the source, target, and the direction of the arrow.
"""

MFR_STAGE_2_PROMPT_TEMPLATE = """
You are an expert AWS Solutions Architect. Your task is to compile the final logical cloud architecture graph from the provided World Model and audio transcript.

You must generalize your reasoning to deduce the true logical "Ground Truth" architecture based on standard AWS patterns, ignoring visual drawing errors but respecting the core system design.

## VALID SERVICES LIST (CRITICAL):
You may ONLY use exact values from this list for the "service" field:
<AWS_SERVICES_PLACEHOLDER>

## MULTILINGUAL TRANSLATION RULE:
Translate all output text fields (notes, graph name, reasoning, etc.) into ENGLISH.

## WORLD MODEL (BASE INPUT):
Use this as your visual inventory.
<WORLD_MODEL_PLACEHOLDER>

## NODE RULES (PRUNING, FUSION, AND ROLE SEPARATION):
1. **Strict Normalization:** The "service" field MUST match exactly with an element from the VALID SERVICES LIST. Map frontend applications, websites, or client endpoints to "UserConsumerWeb".
2. **Numeric Identifiers:** The "id" field of each node MUST be strictly a sequential integer in string format (e.g., "0", "1", "2").
3. **Logical Fusion vs. Separation:** Fuse redundant icons that represent the same logical component. HOWEVER, if the transcript describes distinct instances of the same service serving DIFFERENT architectural roles (e.g., a "Bidding" EC2 cluster vs. a "Tracking" EC2 cluster), keep them as SEPARATE nodes.
4. **Pruning:** Remove generic human actors or purely physical concepts. Keep system entry points.
5. **Note Assimilation:** Extract specific architectural constraints and metrics from the transcript (e.g., "1 million requests per second", "Spot instances", "active-passive") and inject them into the "notes" of the relevant nodes.

## EDGE AND FLOW RULES (LOGICAL ROUTING):
1. **Strict Flow Segmentation (`flow_id`):** Group related architectural actions into distinct logical workflows using an integer `flow_id` (starting at 0).
   - `0`: Usually the primary baseline, ingestion, or synchronous workflow.
   - `1`, `2`, `3`, etc.: Subsequent, decoupled, or asynchronous workflows (e.g., data conversion, analytics querying, background tracking, remediation).
2. **Chronological Sequence (`seq`):**
   - Order events within a flow using string integers ("0", "1", "2").
   - **Parallel Actions:** If a single component triggers MULTIPLE downstream actions simultaneously in the same step, use the prime character to denote parallel paths (e.g., "1" and "1'").
   - **Bidirectionality:** Request/response cycles must be modeled as two edges (e.g., Request is "0", Response is "1").
3. **Edge Types (`type`):**
   - Use "data" for payload transfers, database reads/writes, or heavy data movement.
   - Use "control" for events, triggers, or asynchronous invocations (e.g., an S3 upload triggering a Lambda, EventBridge rules).
"""

# ==========================================
# FASE 1: Agente Modelador
# ==========================================
print("--- FASE 1: Construcción del Modelo del Mundo (Gemini Vision + Pydantic) ---")
ws_world_model_json = LAB_WORKSPACE / "world_model.json"

if ws_world_model_json.exists():
    print(f"⚡ Archivo detectado en caché. Cargando Modelo del Mundo local para ahorrar tokens...")
    with open(ws_world_model_json, "r", encoding="utf-8") as f:
        world_model = json.load(f)
else:
    print("🔍 Generando nuevo Modelo del Mundo con Gemini Vision...")
    formatted_prompt_1 = MFR_STAGE_1_PROMPT_TEMPLATE.replace(
        "<USER_ACTORS_PLACEHOLDER>", user_actors_str
    ).replace(
        "<AWS_SERVICES_PLACEHOLDER>", aws_services_str
    )
    full_prompt_1 = f"{formatted_prompt_1}\n\n## FULL TRANSCRIPT:\n{transcript_text}"

    response_1 = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[
            {
                "parts": [
                    {"text": full_prompt_1},
                    {
                        "inline_data": {
                            "mime_type": "image/jpeg",
                            "data": image_b64,
                        }
                    },
                ]
            }
        ],
        config={
            "response_mime_type": "application/json",
            "response_schema": WorldModelSchema,
        },
    )

    world_model = json.loads(response_1.text)
    ws_world_model_json.write_text(json.dumps(world_model, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"✓ Modelo del Mundo guardado en: {ws_world_model_json}")

# FASE 2: Agente Planificador
print("\n--- FASE 2: Planificación y Generación del Grafo ---")

ws_test_json = LAB_WORKSPACE / "test_analysis.json"
test_graph_path = LAB_WORKSPACE / "test_graph.graphml"

# Lógica de Caché para la Fase 2
if ws_test_json.exists():
    print(f"⚡ Archivo detectado en caché. Cargando Análisis Final local para ahorrar tokens...")
    with open(ws_test_json, "r", encoding="utf-8") as f:
        test_result = json.load(f)
else:
    print("🔍 Generando nuevo Análisis Final con Gemini y Pydantic...")

    # CORRECCIÓN 1: Inyectar la lista de servicios válidos
    formatted_prompt_2 = MFR_STAGE_2_PROMPT_TEMPLATE.replace(
        "<WORLD_MODEL_PLACEHOLDER>", json.dumps(world_model, indent=2, ensure_ascii=False)
    ).replace(
        "<AWS_SERVICES_PLACEHOLDER>", aws_services_str
    )
    full_prompt_2 = f"{formatted_prompt_2}\n\n## FULL TRANSCRIPT:\n{transcript_text}"

    response_2 = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[
            {
                "parts": [
                    {"text": full_prompt_2},
                    {
                        "inline_data": {
                            "mime_type": "image/jpeg",
                            "data": image_b64,
                        }
                    },
                ]
            }
        ],
        config={
            "response_mime_type": "application/json",
            "response_schema": FinalArchitectureSchema, # CORRECCIÓN 2: Aplicar el esquema
        },
    )

    # CORRECCIÓN 3: Carga directa y limpia gracias a Pydantic
    test_result = json.loads(response_2.text)
    ws_test_json.write_text(json.dumps(test_result, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"✓ Análisis de visión final guardado en: {ws_test_json}")

# Construir y exportar el grafo experimental de prueba
test_graph = create_graph_from_cloudscape_json(test_result, video_id=VIDEO_ID)
nx.write_graphml(test_graph, str(test_graph_path))
print(f"✓ Grafo experimental exportado localmente a: {test_graph_path}")

--- FASE 1: Construcción del Modelo del Mundo (Gemini Vision + Pydantic) ---
⚡ Archivo detectado en caché. Cargando Modelo del Mundo local para ahorrar tokens...

--- FASE 2: Planificación y Generación del Grafo ---
🔍 Generando nuevo Análisis Final con Gemini y Pydantic...
✓ Análisis de visión final guardado en: /home/stemjara/Projects/AWS-Architecture/whiteboard_selection_lab/lab_workspace/l0mvXogANE4/test_analysis.json


✓ Graph created: 9 nodes, 11 edges

✓ Grafo experimental exportado localmente a: /home/stemjara/Projects/AWS-Architecture/whiteboard_selection_lab/lab_workspace/l0mvXogANE4/test_graph.graphml


## Fase de Experimentación y Evaluación de Prompts (Gemini API)

Las celdas de abajo te permiten probar en vivo la API de Gemini Vision con la pizarra seleccionada localmente en tu laboratorio y la transcripción detectada.

Se utiliza un flujo de dos agentes (Modeler + Planner) en el que se fuerza una **regla de traducción multilingüe al inglés** para que las notas y descripciones del grafo queden homogeneizadas con el formato de Ground Truth, a pesar de que el video original sea en español.

In [ ]:
# ==========================================
# 7. DEFINICIÓN DE PROMPTS Y EJECUCIÓN CON GEMINI API
# ==========================================
import base64
import json
import csv
import re
from google import genai
from config.settings import GEMINI_API_KEY, GEMINI_MODEL

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY no configurado. Revisa tu archivo .env")

client = genai.Client(api_key=GEMINI_API_KEY)

# 1. Cargar y codificar imagen de pizarra local en base64
ws_whiteboard = LAB_WORKSPACE / "best_whiteboard.jpg"
if not ws_whiteboard.exists():
    # Intenta copiar desde frames_new/video_id_pizarra/best_whiteboard.jpg
    pizarra_dir = LAB_DIR / 'frames_new' / f"{VIDEO_ID}_pizarra"
    shutil.copy2(pizarra_dir / "best_whiteboard.jpg", ws_whiteboard)

image_bytes = ws_whiteboard.read_bytes()
image_b64 = base64.b64encode(image_bytes).decode("utf-8")

# 2. Cargar transcripción local
transcript_file = TRANSCRIPTIONS_DIR / f"{VIDEO_ID}_transcript.json"
with open(transcript_file, "r", encoding="utf-8") as f:
    segments = json.load(f)
transcript_text = " ".join(s.get("text", "").strip() for s in segments)

# 3. Cargar catálogo de servicios y actores
services_csv = PROJECT_ROOT / "data" / "cloudscape_gt" / "services.csv"
aws_services_list, user_actors_list = [], []
with open(services_csv, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        name = row.get("name", "").strip()
        capability = row.get("capability", "").strip().lower()
        if not name:
            continue
        if capability == "user" or name.startswith("User"):
            user_actors_list.append(name)
        else:
            aws_services_list.append(name)

aws_services_str = ", ".join(sorted(aws_services_list))
user_actors_str = ", ".join(sorted(user_actors_list))

# 4. Definición de prompts (Modeler y Planner) con soporte de traducción al inglés
MFR_STAGE_1_PROMPT_TEMPLATE = """You are an expert in visual extraction of cloud architectures. Your task is to analyze the provided whiteboard diagram and audio transcript to build a structured "World Model" representing the components and their visual groupings.

Do NOT generate the final graph. Focus strictly on identifying the present entities, how they are visually grouped, and what lines connect them.

## ENTITY IDENTIFICATION AND GROUPING RULES:
1. Identify all active systems, databases, cloud services, and actors VISIBLE on the whiteboard.
2. Identify BOXES or dashed containers. If a service (e.g., SAP) is drawn inside another service or box (e.g., EC2), register both and strictly note this containment in the "rationale".
3. Use valid AWS services from this list: <AWS_SERVICES_PLACEHOLDER>.
4. Identify actors/users from this list: <USER_ACTORS_PLACEHOLDER>. Map on-premises/external systems to "ThirdParty" and actors to "User".
5. Do NOT include transient files, packages, or disk images (like AMIs) as entities.

## VISUAL CONNECTION RULES:
1. Register ALL lines or arrows explicitly drawn between entities on the whiteboard.
2. Note the source, target, and the direction of the arrow.

## JSON FORMAT RULES (CRITICAL):
1. Return ONLY a valid JSON object. Do NOT include markdown code blocks (```json ... ```) or introductory text.
2. NEVER use unescaped double quotes inside the string values. If you need to emphasize, use single quotes ('text').

## EXACT OUTPUT FORMAT:
{
  "entities": [
    {"service": "AWS Service Name", "name": "Written label", "type": "AWS_Service|User|ThirdParty", "rationale": "Visual justification and if it is inside any boundary box"}
  ],
  "visual_connections": [
    {"source_label": "Source label", "target_label": "Target label", "arrow_direction": "source -> target", "description": "What this line visually represents"}
  ]
}
"""


MFR_STAGE_2_PROMPT_TEMPLATE = """
Eres un Arquitecto de Soluciones AWS experto. Tu tarea es compilar el grafo de la arquitectura cloud lógica final a partir del Modelo del Mundo proporcionado, la imagen de la pizarra y la transcripción de audio.

Debes ir más allá de la representación visual literal y construir el "Ground Truth" lógico.

## LISTA DE SERVICIOS VÁLIDOS (CRÍTICO):
Solo puedes usar valores exactos de esta lista para el campo "service":
<AWS_SERVICES_PLACEHOLDER>

## REGLA DE TRADUCCIÓN MULTILINGÜE:
Traduce todos los campos de texto de salida (notes, name del grafo) al INGLÉS, sin importar el idioma original.

## MODELO DEL MUNDO (INPUT BASE):
Úsalo como inventario, pero aplícale las reglas de arquitectura pura.
<WORLD_MODEL_PLACEHOLDER>

## REGLAS PARA LOS NODOS (PODA, FUSIÓN Y NORMALIZACIÓN):
1. **Normalización Estricta:** El campo "service" DEBE coincidir exactamente con un elemento de la LISTA DE SERVICIOS VÁLIDOS.
2. **Identificadores Numéricos:** El campo "id" de cada nodo DEBE ser estrictamente un número entero secuencial en formato string (ej. "0", "1", "2").
3. **Fusión de Servicios (ANTI-ALUCINACIÓN CRÍTICA):** Debes fusionar OBLIGATORIAMENTE 'CW Events', 'EventBridge' y 'CloudWatch' en un ÚNICO nodo padre cuyo campo service sea "CloudWatch". ESTÁ ESTRICTAMENTE PROHIBIDO crear un nodo independiente para EventBridge o CW Events.
4. **Asimilación de Notas:** NO crees nodos para canales de notificación (Email, ITSM). Asimila esa información en el campo "notes" del nodo que dispara la acción. Extrae y cita textualmente detalles operativos del audio.


## REGLAS PARA LAS ARISTAS Y FLUJOS (ENRUTAMIENTO LÓGICO):
1. **Pivote Bidireccional de Aplicación:** Conecta la infraestructura (EC2) y la aplicación (SAP) de forma bidireccional (EC2 <-> SAP).
2. **Segmentación Estricta de Flujos (`flow_id`, `seq` y `type`):**
   - `flow_id` (Entero):
     * `0`: Flujo base/infraestructura (EC2 <-> SAP).
     * `1`: Flujo de alerta/detección. **OBLIGATORIO: Este flujo DEBE nacer de la aplicación, no del servidor (Ruta estricta: SAP -> CloudWatch -> SNS).**
     * `2`: Flujo de remediación (Ruta estricta: SNS -> Lambda -> SSM -> SAP).
   - `seq` (String): Orden cronológico dentro de cada flujo.
     * Inicia en "0" para la primera acción del flujo, y aumenta a "1", "2".
     * Para el flujo bidireccional (flow_id: 0), el envío es "0" (EC2->SAP) y el retorno es "1" (SAP->EC2).
   - `type` (String): USA EXCLUSIVAMENTE "data" para TODAS las aristas sin excepción.

## FORMATO DE SALIDA (JSON PURO SIN MARKDOWN):
{
  "step_by_step_reasoning": "...",
  "graph": {
    "name": "<título>",
    "link": "",
    "categories": "control",
    "graph_usable": true,
    "notes": "..."
  },
  "nodes": [
    {"id": "0", "service": "Nombre EXACTO de la lista de servicios", "name": "Nombre lógico/etiqueta", "notes": "..."}
  ],
  "edges": [
    {"source": "id", "target": "id", "flow_id": 0, "seq": "0", "type": "control", "notes": "..."}
  ]
}
"""

def clean_json_text(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1]
        if "```" in text:
            text = text[:text.rfind("```")]
    text = text.strip()
    text = re.sub(r',\s*([\]}])', r'\1', text)
    return text

# FASE 1: Agente Modelador
print("--- FASE 1: Construcción del Modelo del Mundo (Gemini Vision) ---")
formatted_prompt_1 = MFR_STAGE_1_PROMPT_TEMPLATE.replace(
    "<USER_ACTORS_PLACEHOLDER>", user_actors_str
).replace(
    "<AWS_SERVICES_PLACEHOLDER>", aws_services_str
)
full_prompt_1 = f"{formatted_prompt_1}\n\n## FULL TRANSCRIPT:\n{transcript_text}"

response_1 = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=[
        {
            "parts": [
                {"text": full_prompt_1},
                {
                    "inline_data": {
                        "mime_type": "image/jpeg",
                        "data": image_b64,
                    }
                },
            ]
        }
    ],
    config={"response_mime_type": "application/json"},
)

world_model = json.loads(clean_json_text(response_1.text))
ws_world_model_json = LAB_WORKSPACE / "world_model.json"
ws_world_model_json.write_text(json.dumps(world_model, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"✓ Modelo del Mundo guardado en: {ws_world_model_json}")

# FASE 2: Agente Planificador
print("\n--- FASE 2: Planificación y Generación del Grafo ---")
formatted_prompt_2 = MFR_STAGE_2_PROMPT_TEMPLATE.replace(
    "<WORLD_MODEL_PLACEHOLDER>", json.dumps(world_model, indent=2, ensure_ascii=False)
)
full_prompt_2 = f"{formatted_prompt_2}\n\n## FULL TRANSCRIPT:\n{transcript_text}"

response_2 = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=[
        {
            "parts": [
                {"text": full_prompt_2},
                {
                    "inline_data": {
                        "mime_type": "image/jpeg",
                        "data": image_b64,
                    }
                },
            ]
        }
    ],
    config={"response_mime_type": "application/json"},
)

test_result = json.loads(clean_json_text(response_2.text))
ws_test_json = LAB_WORKSPACE / "test_analysis.json"
ws_test_json.write_text(json.dumps(test_result, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"✓ Análisis de visión final guardado en: {ws_test_json}")

# Construir y exportar el grafo experimental de prueba
test_graph = create_graph_from_cloudscape_json(test_result, video_id=VIDEO_ID)
test_graph_path = LAB_WORKSPACE / "test_graph.graphml"
nx.write_graphml(test_graph, str(test_graph_path))
print(f"✓ Grafo experimental exportado localmente a: {test_graph_path}")

In [ ]:
# ==========================================
# 7. DEFINICIÓN DE PROMPTS Y EJECUCIÓN CON GEMINI API
# ==========================================
import base64
import json
import csv
import re
from google import genai
from config.settings import GEMINI_API_KEY, GEMINI_MODEL

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY no configurado. Revisa tu archivo .env")

client = genai.Client(api_key=GEMINI_API_KEY)

# 1. Cargar y codificar imagen de pizarra local en base64
ws_whiteboard = LAB_WORKSPACE / "best_whiteboard.jpg"
if not ws_whiteboard.exists():
    # Intenta copiar desde frames_new/video_id_pizarra/best_whiteboard.jpg
    pizarra_dir = LAB_DIR / 'frames_new' / f"{VIDEO_ID}_pizarra"
    shutil.copy2(pizarra_dir / "best_whiteboard.jpg", ws_whiteboard)

image_bytes = ws_whiteboard.read_bytes()
image_b64 = base64.b64encode(image_bytes).decode("utf-8")

# 2. Cargar transcripción local
transcript_file = TRANSCRIPTIONS_DIR / f"{VIDEO_ID}_transcript.json"
with open(transcript_file, "r", encoding="utf-8") as f:
    segments = json.load(f)
transcript_text = " ".join(s.get("text", "").strip() for s in segments)

# 3. Cargar catálogo de servicios y actores
services_csv = PROJECT_ROOT / "data" / "cloudscape_gt" / "services.csv"
aws_services_list, user_actors_list = [], []
with open(services_csv, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        name = row.get("name", "").strip()
        capability = row.get("capability", "").strip().lower()
        if not name:
            continue
        if capability == "user" or name.startswith("User"):
            user_actors_list.append(name)
        else:
            aws_services_list.append(name)

aws_services_str = ", ".join(sorted(aws_services_list))
user_actors_str = ", ".join(sorted(user_actors_list))

# 4. Definición de prompts (Modeler y Planner) con soporte de traducción al inglés
MFR_STAGE_1_PROMPT_TEMPLATE = """Eres un experto en extracción visual de arquitecturas cloud. Tu tarea es analizar el diagrama de la pizarra proporcionado y la transcripción de audio para construir un "Modelo del Mundo" estructurado que represente los componentes y sus agrupaciones.

NO generes el grafo final. Enfócate estrictamente en identificar las entidades presentes, cómo están agrupadas visualmente y qué líneas las conectan.

## REGLAS DE IDENTIFICACIÓN DE ENTIDADES Y AGRUPACIONES:
1. Identifica todos los sistemas activos, bases de datos, servicios cloud y actores VISIBLES en la pizarra.
2. Identifica RECUADROS o contenedores punteados. Si un servicio (ej. SAP) está dibujado dentro de otro servicio o recuadro (ej. EC2), regístralos ambos y anota esta contención en el "rationale".
3. Usa servicios válidos de AWS de esta lista: <AWS_SERVICES_PLACEHOLDER>.
4. Identifica actores/usuarios de esta lista: <USER_ACTORS_PLACEHOLDER>. Mapea sistemas on-premises a "ThirdParty" y actores a "User".
5. NO incluyas archivos transitorios, paquetes o imágenes de disco (como AMIs) como entidades.

## REGLAS DE CONEXIONES VISUALES:
1. Registra TODAS las líneas o flechas dibujadas explícitamente entre las entidades en la pizarra.
2. Anota el origen, el destino y la dirección de la flecha.

## REGLAS DE FORMATO JSON (CRÍTICAS):
1. Devuelve ÚNICAMENTE un objeto JSON válido. NO incluyas bloques de código markdown (```json ... ```) ni texto introductorio.
2. NUNCA uses comillas dobles dentro de los valores de texto de las claves. Si necesitas enfatizar, usa comillas simples ('texto').

## FORMATO DE SALIDA EXACTO:
{
  "entities": [
    {"service": "Nombre del servicio AWS", "name": "Etiqueta escrita", "type": "AWS_Service|User|ThirdParty", "rationale": "Justificación visual y si está dentro de algún recuadro"}
  ],
  "visual_connections": [
    {"source_label": "Etiqueta origen", "target_label": "Etiqueta destino", "arrow_direction": "origen -> destino", "description": "Qué representa visualmente esta línea"}
  ]
}
"""

MFR_STAGE_2_PROMPT_TEMPLATE = """
Eres un Arquitecto de Soluciones AWS experto. Tu tarea es compilar el grafo de la arquitectura cloud lógica final a partir del Modelo del Mundo proporcionado, la imagen de la pizarra y la transcripción de audio.

Debes ir más allá de la representación visual literal y construir el "Ground Truth" lógico.

## LISTA DE SERVICIOS VÁLIDOS (CRÍTICO):
Solo puedes usar valores exactos de esta lista para el campo "service":
<AWS_SERVICES_PLACEHOLDER>

## REGLA DE TRADUCCIÓN MULTILINGÜE:
Traduce todos los campos de texto de salida (notes, name del grafo) al INGLÉS, sin importar el idioma original.

## MODELO DEL MUNDO (INPUT BASE):
Úsalo como inventario, pero aplícale las reglas de arquitectura pura.
<WORLD_MODEL_PLACEHOLDER>

## REGLAS PARA LOS NODOS (PODA, FUSIÓN Y NORMALIZACIÓN):
1. **Normalización Estricta:** El campo "service" DEBE coincidir exactamente con un elemento de la LISTA DE SERVICIOS VÁLIDOS.
2. **Identificadores Numéricos:** El campo "id" de cada nodo DEBE ser estrictamente un número entero secuencial en formato string (ej. "0", "1", "2").
3. **Fusión de Servicios (ANTI-ALUCINACIÓN CRÍTICA):** Debes fusionar OBLIGATORIAMENTE 'CW Events', 'EventBridge' y 'CloudWatch' en un ÚNICO nodo padre cuyo campo service sea "CloudWatch". ESTÁ ESTRICTAMENTE PROHIBIDO crear un nodo independiente para EventBridge o CW Events.
4. **Asimilación de Notas:** NO crees nodos para canales de notificación (Email, ITSM). Asimila esa información en el campo "notes" del nodo que dispara la acción. Extrae y cita textualmente detalles operativos del audio.


## REGLAS PARA LAS ARISTAS Y FLUJOS (ENRUTAMIENTO LÓGICO):
1. **Pivote Bidireccional de Aplicación:** Conecta la infraestructura (EC2) y la aplicación (SAP) de forma bidireccional (EC2 <-> SAP).
2. **Segmentación Estricta de Flujos (`flow_id`, `seq` y `type`):**
   - `flow_id` (Entero):
     * `0`: Flujo base/infraestructura (EC2 <-> SAP).
     * `1`: Flujo de alerta/detección. **OBLIGATORIO: Este flujo DEBE nacer de la aplicación, no del servidor (Ruta estricta: SAP -> CloudWatch -> SNS).**
     * `2`: Flujo de remediación (Ruta estricta: SNS -> Lambda -> SSM -> SAP).
   - `seq` (String): Orden cronológico dentro de cada flujo.
     * Inicia en "0" para la primera acción del flujo, y aumenta a "1", "2".
     * Para el flujo bidireccional (flow_id: 0), el envío es "0" (EC2->SAP) y el retorno es "1" (SAP->EC2).
   - `type` (String): USA EXCLUSIVAMENTE "data" para TODAS las aristas sin excepción.

## FORMATO DE SALIDA (JSON PURO SIN MARKDOWN):
{
  "step_by_step_reasoning": "...",
  "graph": {
    "name": "<título>",
    "link": "",
    "categories": "control",
    "graph_usable": true,
    "notes": "..."
  },
  "nodes": [
    {"id": "0", "service": "Nombre EXACTO de la lista de servicios", "name": "Nombre lógico/etiqueta", "notes": "..."}
  ],
  "edges": [
    {"source": "id", "target": "id", "flow_id": 0, "seq": "0", "type": "control", "notes": "..."}
  ]
}
"""

def clean_json_text(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1]
        if "```" in text:
            text = text[:text.rfind("```")]
    text = text.strip()
    text = re.sub(r',\s*([\]}])', r'\1', text)
    return text

# FASE 1: Agente Modelador
print("--- FASE 1: Construcción del Modelo del Mundo (Gemini Vision) ---")
formatted_prompt_1 = MFR_STAGE_1_PROMPT_TEMPLATE.replace(
    "<USER_ACTORS_PLACEHOLDER>", user_actors_str
).replace(
    "<AWS_SERVICES_PLACEHOLDER>", aws_services_str
)
full_prompt_1 = f"{formatted_prompt_1}\n\n## FULL TRANSCRIPT:\n{transcript_text}"

response_1 = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=[
        {
            "parts": [
                {"text": full_prompt_1},
                {
                    "inline_data": {
                        "mime_type": "image/jpeg",
                        "data": image_b64,
                    }
                },
            ]
        }
    ],
    config={"response_mime_type": "application/json"},
)

world_model = json.loads(clean_json_text(response_1.text))
ws_world_model_json = LAB_WORKSPACE / "world_model.json"
ws_world_model_json.write_text(json.dumps(world_model, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"✓ Modelo del Mundo guardado en: {ws_world_model_json}")

# FASE 2: Agente Planificador
print("\n--- FASE 2: Planificación y Generación del Grafo ---")
formatted_prompt_2 = MFR_STAGE_2_PROMPT_TEMPLATE.replace(
    "<WORLD_MODEL_PLACEHOLDER>", json.dumps(world_model, indent=2, ensure_ascii=False)
)
full_prompt_2 = f"{formatted_prompt_2}\n\n## FULL TRANSCRIPT:\n{transcript_text}"

response_2 = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=[
        {
            "parts": [
                {"text": full_prompt_2},
                {
                    "inline_data": {
                        "mime_type": "image/jpeg",
                        "data": image_b64,
                    }
                },
            ]
        }
    ],
    config={"response_mime_type": "application/json"},
)

test_result = json.loads(clean_json_text(response_2.text))
ws_test_json = LAB_WORKSPACE / "test_analysis.json"
ws_test_json.write_text(json.dumps(test_result, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"✓ Análisis de visión final guardado en: {ws_test_json}")

# Construir y exportar el grafo experimental de prueba
test_graph = create_graph_from_cloudscape_json(test_result, video_id=VIDEO_ID)
test_graph_path = LAB_WORKSPACE / "test_graph.graphml"
nx.write_graphml(test_graph, str(test_graph_path))
print(f"✓ Grafo experimental exportado localmente a: {test_graph_path}")

## Resumen Comparativo de Rendimiento Lado a Lado

La celda final realiza una comparación detallada de rendimiento calculando la precisión, recall y F1 de los servicios y las conexiones de la nueva prueba frente a las ejecuciones estándar y parsimoniosas cached, mostrándolo en una tabla de Pandas interactiva.

In [18]:
# ==========================================
# 8. COMPARACIÓN DETALLADA LADO A LADO
# ==========================================
import pandas as pd
from scripts.utils.evaluate_graphs import evaluate_pair, load_services_catalog

# Cargar catálogo de servicios
catalog_dict = load_services_catalog(services_csv)

# Cargar grafos
gt_path = PROJECT_ROOT / "data" / "cloudscape_gt" / f"{VIDEO_ID}.graphml"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
gt_g = nx.read_graphml(str(gt_path))
test_g = nx.read_graphml(str(test_graph_path))

# Rutas locales de grafos en data/
ws_std_graph = RAW_DIR / f"{VIDEO_ID}.graphml"
if not ws_std_graph.exists():
    ws_std_graph = PROJECT_ROOT / "data" / "graphs" / f"{VIDEO_ID}.graphml"

ws_parsi_graph = RAW_DIR / f"{VIDEO_ID}_parsimonious.graphml"
if not ws_parsi_graph.exists():
    ws_parsi_graph = PROJECT_ROOT / "data" / "graphs_parsimonious" / f"{VIDEO_ID}.graphml"

# Evaluar nueva prueba contra GT
test_eval = evaluate_pair(test_g, gt_g, VIDEO_ID, catalog_dict)

def evaluar_previo(path):
    if path.exists():
        try:
            g = nx.read_graphml(str(path))
            return evaluate_pair(g, gt_g, VIDEO_ID, catalog_dict)
        except Exception as e:
            return {"error": str(e)}
    return None

std_eval = evaluar_previo(ws_std_graph)
parsi_eval = evaluar_previo(ws_parsi_graph)

# Generar filas de DataFrame comparativo
filas = []
def add_row(label, key, is_pct=True):
    fila = {"Métrica": label}
    if key == "gen_nodes":
        fila["Ground Truth"] = gt_g.number_of_nodes()
    elif key == "gen_edges":
        fila["Ground Truth"] = gt_g.number_of_edges()
    else:
        fila["Ground Truth"] = "—"
        
    # Standard
    if std_eval and key in std_eval:
        val = std_eval[key]
        fila["Standard Original"] = f"{val:.1%}" if is_pct else val
    else:
        fila["Standard Original"] = "N/A"
        
    # Parsimonious
    if parsi_eval and key in parsi_eval:
        val = parsi_eval[key]
        fila["Parsimonious Original"] = f"{val:.1%}" if is_pct else val
    else:
        fila["Parsimonious Original"] = "N/A"
        
    # Test
    if key in test_eval:
        val = test_eval[key]
        fila["Nueva Prueba (Test)"] = f"{val:.1%}" if is_pct else val
    else:
        fila["Nueva Prueba (Test)"] = "—"
        
    filas.append(fila)

add_row("Número de Nodos", "gen_nodes", is_pct=False)
add_row("Número de Aristas", "gen_edges", is_pct=False)
add_row("Service F1 (Unique)", "svc_f1")
add_row("Service Precision", "svc_precision")
add_row("Service Recall", "svc_recall")
add_row("Edge F1 (Connections)", "edge_f1")
add_row("Edge Precision", "edge_precision")
add_row("Edge Recall", "edge_recall")

df_resumen = pd.DataFrame(filas)
print("📊 RESUMEN COMPARATIVO DE RENDIMIENTO DE GRAFOS DE PRUEBA:")
display(df_resumen)

# Detalles cualitativos de errores
print("\n🔍 ERRORES O DETALLES EN LA NUEVA PRUEBA (TEST):")
print(f"  • Servicios Faltantes (Omitidos):   {test_eval.get('services_missing', [])}")
print(f"  • Servicios Alucinados (Inventados): {test_eval.get('services_hallucinated', [])}")

if parsi_eval:
    print("\n🔄 COMPARACIÓN DE ERRORES CON PARSIMONIOUS ORIGINAL:")
    print(f"  • Faltaban antes: {parsi_eval.get('services_missing', [])}")
    print(f"  • Faltan ahora:   {test_eval.get('services_missing', [])}")
    print(f"  • Alucinaba antes: {parsi_eval.get('services_hallucinated', [])}")
    print(f"  • Alucina ahora:   {test_eval.get('services_hallucinated', [])}")

📊 RESUMEN COMPARATIVO DE RENDIMIENTO DE GRAFOS DE PRUEBA:


,Métrica,Ground Truth,Standard Original,Parsimonious Original,Nueva Prueba (Test)
0,Número de Nodos,8,9,9,9
1,Número de Aristas,14,11,11,11
2,Service F1 (Unique),—,93.3%,100.0%,100.0%
3,Service Precision,—,87.5%,100.0%,100.0%
4,Service Recall,—,100.0%,100.0%,100.0%
5,Edge F1 (Connections),—,64.0%,88.0%,88.0%
6,Edge Precision,—,72.7%,100.0%,100.0%
7,Edge Recall,—,57.1%,78.6%,78.6%



🔍 ERRORES O DETALLES EN LA NUEVA PRUEBA (TEST):
  • Servicios Faltantes (Omitidos):   []
  • Servicios Alucinados (Inventados): []

🔄 COMPARACIÓN DE ERRORES CON PARSIMONIOUS ORIGINAL:
  • Faltaban antes: []
  • Faltan ahora:   []
  • Alucinaba antes: []
  • Alucina ahora:   []
